# Assignment: LangGraph Agent with Custom Math Functions
**Course:** Building Advanced AI Agents with LangGraph  
**Goal:** Agent that answers general questions via LLM and math queries via custom functions (plus, subtract, multiply, divide)

## Step 1 — Install Dependencies

In [ ]:
!pip install langgraph langchain-groq langchain-core python-dotenv

## Step 2 — Set API Key
Get free key at: https://console.groq.com  
**Never hardcode key in notebook — use environment variable.**

In [ ]:
import os

# Option A: set directly in session (not saved to file)
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"  # replace, do NOT commit

# Option B: load from .env file (recommended)
# from dotenv import load_dotenv
# load_dotenv()  # reads GROQ_API_KEY from .env file

print("API key set:", bool(os.environ.get("GROQ_API_KEY")))

## Step 3 — Import Libraries

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("All imports successful.")

## Step 4 — Define Agent State

In [ ]:
class AgentState(TypedDict):
    """
    State passed between nodes in the graph.
    'messages' accumulates conversation history.
    add_messages reducer appends new messages instead of overwriting.
    """
    messages: Annotated[list, add_messages]

## Step 5 — Define Custom Math Functions (Tools)

In [ ]:
@tool
def plus(a: float, b: float) -> float:
    """Add two numbers together. Use for addition queries."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a. Use for subtraction queries."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together. Use for multiplication queries."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Handles division by zero with a clear error."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b


# Register all tools
tools = [plus, subtract, multiply, divide]

print("Tools defined:", [t.name for t in tools])

## Step 6 — Set Up LLM with Tools Bound

In [ ]:
# LLM: Groq runs llama-3.3-70b for free, very fast
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Binding tools tells the LLM about available functions
# LLM will decide WHEN to call them based on user query
llm_with_tools = llm.bind_tools(tools)

print("LLM ready with", len(tools), "tools bound.")

## Step 7 — Define Graph Nodes

In [ ]:
def agent_node(state: AgentState) -> AgentState:
    """
    Agent node: sends messages to LLM.
    LLM either:
      - Returns tool_calls  → router sends to tools node
      - Returns plain text  → router ends the graph
    """
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ToolNode automatically executes whichever tool the LLM chose
tool_node = ToolNode(tools)

print("Nodes defined: agent_node, tool_node")

## Step 8 — Define Routing Logic

In [ ]:
def should_continue(state: AgentState) -> str:
    """
    Router: checks last LLM message.
    - Has tool_calls  → go to 'tools' node
    - No tool_calls   → END (answer is ready)
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END


print("Router defined.")

## Step 9 — Build the LangGraph

In [ ]:
# Initialize graph with state schema
graph_builder = StateGraph(AgentState)

# Add nodes
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)

# Entry point: always start at agent
graph_builder.set_entry_point("agent")

# Conditional edge: agent → tools OR END
graph_builder.add_conditional_edges(
    "agent",          # from node
    should_continue,  # routing function
    {"tools": "tools", END: END}  # possible destinations
)

# After tools run → back to agent for final response
graph_builder.add_edge("tools", "agent")

# Compile graph
graph = graph_builder.compile()

print("Graph compiled successfully.")

## Step 10 — Visualize Graph Flow
```
User Query
    │
    ▼
[agent node]  ←─────────────────────┐
    │                                │
    ├── tool_calls? ──YES──► [tools node]
    │                                │
    │                         runs plus/subtract/
    │                         multiply/divide
    │                                │
    │                                └────────────┘
    │
    └── no tool_calls? ──────────────► END
                                   (LLM answered directly)
```

## Step 11 — Chat Helper Function

In [ ]:
def chat(query: str) -> str:
    """Send a query to the agent and return the final response."""
    result = graph.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content

## Step 12 — Test Math Queries

In [ ]:
math_queries = [
    "What is 5 plus 3?",
    "How much is 8 divided by 2?",
    "Multiply 7 by 6",
    "Subtract 15 from 40",
    "What is 100 divided by 0?",   # division by zero test
]

print("=" * 50)
print("MATH QUERIES")
print("=" * 50)

for query in math_queries:
    print(f"\nQ: {query}")
    try:
        print(f"A: {chat(query)}")
    except Exception as e:
        print(f"Error: {e}")

## Step 13 — Test General Queries

In [ ]:
general_queries = [
    "What is the capital of France?",
    "Explain what machine learning is in one sentence.",
    "Who wrote the Harry Potter series?",
]

print("=" * 50)
print("GENERAL QUERIES (LLM answers directly — no tool called)")
print("=" * 50)

for query in general_queries:
    print(f"\nQ: {query}")
    print(f"A: {chat(query)}")

## Step 14 — Interactive Chat Loop (Optional)

In [ ]:
# Run this cell for interactive chat. Type 'quit' to stop.
print("Agent ready. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    try:
        response = chat(user_input)
        print(f"Agent: {response}\n")
    except Exception as e:
        print(f"Error: {e}\n")